# Align instances

In [ ]:
from pathlib import Path

root_dir = Path.home() / 'Code/Paper-Dataset'

In [ ]:
import json

with (root_dir / 'final_data' / 'input_new' / 'msd_20251103.json').open('r') as f:
    msd = json.load(f)
msd = {d['instance_id']: d for d in msd}

with (root_dir / 'final_data' / 'input_new' / 'gesund_bund_20251103.json').open('r') as f:
    gesund = json.load(f)
gesund = {d['instance_id']: d for d in gesund}

with (root_dir / 'final_data' / 'input_new' / 'nhs_20251103.json').open('r') as f:
    nhs = json.load(f)
nhs = {d['instance_id']: d for d in nhs}

with (root_dir / 'final_data' / 'input_new' / 'apoum_20251103.json').open('r') as f:
    apoum = json.load(f)
apoum = {d['instance_id']: d for d in apoum}

In [ ]:
from collections import defaultdict

def reverse_index(data):
    record = {name: defaultdict(set) for name in ['ICD-10', 'ICD-10CM', 'MeSH', 'DOID', 'Pschyrembel', 'MSD']}
    for instance_id, r in data.items():
        for name, values in r['external_ids'].items():
            if isinstance(values, list):
                for v in values:
                    try:
                        record[name][v['value']].add(instance_id)
                    except KeyError:
                        print(
                            f'KeyError for instance_id={instance_id} name={name} values={values}'
                        )
    return record

In [ ]:
msd_rev = reverse_index(msd)
gesund_rev = reverse_index(gesund)
nhs_rev = reverse_index(nhs)
apoum_rev = reverse_index(apoum)

### expand by dictionary indices

In [ ]:
from tqdm.auto import tqdm
records = []
for main_key in ['ICD-10CM', 'ICD-10', 'ICD-10CM', 'MeSH', 'DOID', 'Pschyrembel']:
    ids = list(msd_rev[main_key].keys()) + list(gesund_rev[main_key].keys()) + list(nhs_rev[main_key].keys()) + list(apoum_rev[main_key].keys()) 
    
    for _id in tqdm(ids):
        record = {name: set([]) for name in [
            'ICD-10', 'ICD-10CM', 'MeSH', 'DOID', 'Pschyrembel', 'MSD', 'gesund', 'NHS', 'apoum'
        ]}
        record[main_key].add(_id)
    
        for key, rev_dict in [('gesund', gesund_rev), ('NHS', nhs_rev), ('apoum', apoum_rev)]:
            for v in rev_dict[main_key][_id]:
                record[key].add(v)
    
        for data_name, data in [('gesund', gesund), ('NHS', nhs), ('apoum', apoum)]:
            for data_instance_id in record[data_name]:
                for name, values in data[data_instance_id]['external_ids'].items():
                    if isinstance(values, list):
                        for v in values:
                            record[name].add(v['value'])
    
        for name in ['ICD-10', 'ICD-10CM', 'MeSH', 'DOID', 'Pschyrembel', 'MSD']:
            for rev_key in record[name]:
                for key, rev_dict in [('gesund', gesund_rev), ('NHS', nhs_rev), ('apoum', apoum_rev)]:
                    for v in rev_dict[name][rev_key]:
                        record[key].add(v)
    
        records.append(record)

### expand by dataset indices

In [ ]:
from tqdm.auto import tqdm

datasets = [('MSD', msd, msd_rev), ('gesund', gesund, gesund_rev), ('NHS', nhs, nhs_rev), ('apoum', apoum, apoum_rev)]
for i, (main_key, data, _) in enumerate(datasets):
    for instance_id, r in tqdm(data.items()):
        record = {name: set([]) for name in [
            'ICD-10', 'ICD-10CM', 'MeSH', 'DOID', 'Pschyrembel', 'MSD', 'gesund', 'NHS', 'apoum'
        ]}
        record[main_key].add(instance_id)
    
        for name, values in r['external_ids'].items():
            if isinstance(values, list):
                for v in values:
                    record[name].add(v['value'])
    
        for name in ['ICD-10', 'ICD-10CM', 'MeSH', 'DOID', 'Pschyrembel', 'MSD']:
            for rev_key in record[name]:
                for key, _, rev_dict in [datasets[j] for j in range(len(datasets)) if j != i]:
                    for v in rev_dict[name][rev_key]:
                        record[key].add(v)
    
        records.append(record)

### marge down

In [ ]:
from collections import Counter
data2record = {name: defaultdict(set) for name in ['MSD', 'gesund', 'NHS', 'apoum']}
for i, r in enumerate(records):
    for name in ['MSD', 'gesund', 'NHS', 'apoum']:
        for v in r[name]:
            data2record[name][v].add(i)

len(data2record['MSD']), len(data2record['apoum']), len(data2record['gesund']), len(data2record['NHS'])

In [ ]:
datasets = [('MSD', msd, msd_rev), ('gesund', gesund, gesund_rev), ('NHS', nhs, nhs_rev), ('apoum', apoum, apoum_rev)]
all_external_ids = {}
for name, data, _ in datasets:
    external_ids = {}
    for instance_id, r in tqdm(data.items()):
        tmp = defaultdict(set)
        for i in data2record[name][instance_id]:
            if instance_id in records[i][name]:
                for key, values in records[i].items():
                    for v in values:
                        tmp[key].add(v)
        external_ids[instance_id] = tmp
    all_external_ids[name] = external_ids

In [ ]:
def set_default(obj):
    if isinstance(obj, set):
        return list(obj)
    raise TypeError

with (root_dir / 'external_ids.json').open('w') as f:
    json.dump(all_external_ids, f, default=set_default, indent=4)

### save in excel

In [ ]:
import pandas as pd

def merge(df, key):
    instances = []
    for ids, group in df.groupby(key):
        instance = defaultdict(set)
        for name, row in group.iterrows():
            for h in header:
                if isinstance(row[f'{h}_id'], str):
                    for v in row[f'{h}_id'].split('; '):
                        instance[f'{h}_id'].add(v)
                    if f'{h}_title' in row:
                        for v in row[f'{h}_title'].split('; '):
                            instance[f'{h}_title'].add(v)
        instances.append({k: '; '.join(v).strip(';').strip() if len(v) > 0 else '' for k, v in instance.items()})
    return pd.DataFrame.from_records(instances)

In [ ]:
import pandas as pd

datasets = {'MSD': (msd, 'en', 'professional'), 'gesund': (gesund, 'en', 'amateur'), 'NHS': (nhs, 'en', 'amateur'), 'apoum': (apoum, 'de', 'amateur')}
header = ['MSD', 'gesund', 'NHS', 'apoum', 'ICD-10', 'ICD-10CM', 'MeSH', 'DOID', 'Pschyrembel']
dfs = {}
for name in ['MSD', 'gesund', 'NHS', 'apoum']:
    instances = []
    for instance_id, alignment in sorted(all_external_ids[name].items(), key=lambda x: x[0]):
        instance = {}
        for h in header:
            align = sorted(alignment[h], key=lambda x: int(x) if h in ['gesund', 'apoum'] else x)
            instance[f'{h}_id'] = '; '.join(align)
            if h in datasets:
                data, lang, ed = datasets[h]
                instance[f'{h}_title'] = '; '.join([data[i][lang][ed]['topic']['name'] for i in align])
        instances.append(instance)
    df = pd.DataFrame.from_records(instances)
    dfs[name] = merge(df, f'{name}_id')
    print(name, len(dfs[name]))

In [ ]:
with pd.ExcelWriter(root_dir / 'instance_alignments_ver2.xlsx') as writer:
    for name, df in dfs.items():
        df.to_excel(writer, sheet_name=name, index=False)

In [ ]:
dfs['apoum'].to_csv(root_dir / 'instance_alignments_apoum.tsv', sep="\t", index=False)

In [ ]:
dfs['NHS'].to_csv(root_dir / 'instance_alignments_nhs.tsv', sep="\t", index=False)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from helpers import load_data_tables
msd_table, gesund_table, nhs_table, apoum_table = load_data_tables()

In [ ]:
from helpers import check_by_similarity
from tqdm.auto import tqdm

for instance_id, r in tqdm(apoum.items()):
    emb = {}
    found = apoum_table['de'].metadata_search(instance_id=instance_id, audience="amateur")
    emb['de'] = found[0]['embedding']
    emb['en'] = found[0]['embedding']

    for name, dict_table in [
        ('gesund', gesund_table),
        ('NHS', nhs_table),
        ('MSD', msd_table),
    ]:
        if len(all_external_ids['apoum'][instance_id][name]) > 0:
            ids = list(all_external_ids['apoum'][instance_id][name])
            distances = check_by_similarity(dict_table, emb, ids, key="instance_id", limit=25)
            r['external_ids'][name] = [{'value': k, 'distance': v} for k, v in distances.items()] if len(distances) > 0 else None

In [ ]:
for instance_id, r in tqdm(nhs.items()):
    emb = {}
    found = nhs_table['en'].metadata_search(instance_id=instance_id, audience="amateur")
    emb['de'] = found[0]['embedding']
    emb['en'] = found[0]['embedding']

    for name, dict_table in [
        ('gesund', gesund_table),
        ('apoum', apoum_table),
        ('MSD', msd_table),
    ]:
        if len(all_external_ids['NHS'][instance_id][name]) > 0:
            ids = list(all_external_ids['NHS'][instance_id][name])
            distances = check_by_similarity(dict_table, emb, ids, key="instance_id", limit=25)
            r['external_ids'][name] = [{'value': k, 'distance': v} for k, v in distances.items()] if len(distances) > 0 else None

In [ ]:
for instance_id, r in tqdm(gesund.items()):
    emb = {}
    for lang, table in gesund_table.items():
        found = table.metadata_search(instance_id=instance_id, audience="amateur")
        emb[lang] = found[0]['embedding']

    for name, dict_table in [
        ('NHS', nhs_table),
        ('apoum', apoum_table),
        ('MSD', msd_table),
    ]:
        if len(all_external_ids['gesund'][instance_id][name]) > 0:
            ids = list(all_external_ids['gesund'][instance_id][name])
            distances = check_by_similarity(dict_table, emb, ids, key="instance_id", limit=25)
            r['external_ids'][name] = [{'value': k, 'distance': v} for k, v in distances.items()] if len(distances) > 0 else None

In [ ]:
for instance_id, r in tqdm(msd.items()):
    emb = {}
    for lang, table in msd_table.items():
        found = table.metadata_search(instance_id=instance_id, audience="amateur")
        try:
            emb[lang] = found[0]['embedding']
        except IndexError:
            continue

    for name, dict_table in [
        ('NHS', nhs_table),
        ('apoum', apoum_table),
        ('gesund', gesund_table),
    ]:
        if len(all_external_ids['MSD'][instance_id][name]) > 0:
            ids = list(all_external_ids['MSD'][instance_id][name])
            distances = check_by_similarity(dict_table, emb, ids, key="instance_id", limit=25)
            r['external_ids'][name] = [{'value': k, 'distance': v} for k, v in distances.items()] if len(distances) > 0 else None

In [ ]:
with (root_dir / 'data/apoum_20251103.json').open('w') as f:
    json.dump([r for _, r in apoum.items()], f, indent=4)

with (root_dir / 'data/nhs_20251103.json').open('w') as f:
    json.dump([r for _, r in nhs.items()], f, indent=4)

with (root_dir / 'data/gesund_bund_20251103.json').open('w') as f:
    json.dump([r for _, r in gesund.items()], f, indent=4)

with (root_dir / 'data/msd_20251103.json').open('w') as f:
    json.dump([r for _, r in msd.items()], f, indent=4)